In [36]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import gurobipy as gp
from gurobipy import GRB

# Data initialization

In [45]:
# initialize relevant data
current_filepath = Path('.').resolve().parent
data_path = current_filepath / 'raw-data'

branches_df = pd.read_csv(filepath_or_buffer=data_path / 'branches.csv', skiprows=1)
buses_df = pd.read_csv(filepath_or_buffer=data_path / 'buses.csv', skiprows=1)
ptdf_df = pd.read_csv(filepath_or_buffer=data_path / 'ptdf.csv', skiprows=1)
sf_contingencies_df = pd.read_csv(filepath_or_buffer=data_path / 'sf-contingencies.csv', skiprows=1)
generators_df = pd.read_csv(filepath_or_buffer=data_path / 'generators.csv') # This was produced by Gemini 3 Pro

In [46]:
generator_min = generators_df['Pmin'].to_numpy()
generator_max = generators_df['Pmax'].to_numpy()
generator_cost = generators_df['Cost'].to_numpy()
generator_node = generators_df['Bus Number'].to_numpy()

nominal_demand = buses_df['Load MW'].replace([np.nan], 0).to_numpy()
load_node = buses_df['Number'].to_numpy()
load_node = load_node[nominal_demand != 0]
nominal_demand = nominal_demand[nominal_demand != 0]

G = generator_cost.shape[0]
D = nominal_demand.shape[0]

thermal_flow_lim = branches_df['Lim MVA A'].to_numpy()

power_balance_cons_vec = np.concatenate((np.ones((G, 1)), -1*np.ones((D, 1)))).T

## Injection Mapping

In [47]:
gen_node_map = np.zeros((buses_df.shape[0], G))
gen_node_map[generator_node - 1, np.arange(G)] = 1
load_node_map = np.zeros((buses_df.shape[0], load_node.shape[0]))
load_node_map[load_node - 1, np.arange(load_node.shape[0])] = -1

injection_map = np.hstack((gen_node_map, load_node_map))

## Shift Factor to Thermal Flow Mapping

In [48]:
# Create a mapping from line to thermal flow lim
thermal_flow_mapping = branches_df[['From Name', 'To Number', 'Lim MVA A']]
sf_no_contingency = ptdf_df.loc[:, '1  TO  2 CKT 1':'21  TO  22 CKT 1']

# match the thermal flow lim to the shift factor values
sf_names = sf_no_contingency.columns.values
thermal_flow_lim = np.zeros((sf_names.shape[0]))
for (ind, line) in enumerate(sf_names):
    line_string = line.split(' ')
    from_no = int(line_string[0])
    to_no = int(line_string[4])
    thermal_flow_lim[ind] = thermal_flow_mapping.query('`From Name` == @from_no and `To Number` == @to_no')['Lim MVA A'].iloc[0]

sf_no_contingency = sf_no_contingency.to_numpy().T

In [49]:
sf_contingency = sf_contingencies_df.loc[:, '9 (MONITOR_1_1-2_CONTINGENCY_10_8-9)':'304 (MONITOR_9_7-8_CONTINGENCY_8_5-10)']

# match the thermal flow lim to the shift factor contingency values
contingency_names = sf_contingency.columns.values
thermal_flow_lim_contingencies = np.zeros((contingency_names.shape[0]))
for (ind, contingency) in enumerate(contingency_names):
    line_string = contingency.split('_')[2]
    from_no = int(line_string.split('-')[0])
    to_no = int(line_string.split('-')[1])
    thermal_flow_lim_contingencies[ind] = thermal_flow_mapping.query('`From Name` == @from_no and `To Number` == @to_no')['Lim MVA A'].iloc[0]

sf_contingency = sf_contingency.to_numpy().T

In [50]:
base_case_trans_cons_mat = sf_no_contingency @ injection_map
contingency_trans_cons_mat = sf_contingency @ injection_map

## Model Set-up

In [51]:
lp_model = gp.Model()
generators = lp_model.addMVar(G, name='generators')
lp_model.setMObjective(Q=None, c=generator_cost, constant=0, sense=GRB.MINIMIZE)
loads = lp_model.addMVar(D, name='load_served', lb=0.0)
power_balance_cons = lp_model.addMConstr(power_balance_cons_vec, gp.hstack((generators, loads)), '=', [0], 'power balance')
generator_min_cons = lp_model.addMConstr(np.eye(G), generators, '>=', generator_min, 'min generator capacity')
generator_max_cons = lp_model.addMConstr(np.eye(G), generators, '<=', generator_max, 'max generator capacity')
load_limits_cons = lp_model.addMConstr(np.eye(D), loads, '<=', nominal_demand, 'max load demand')
base_case_cons_min = lp_model.addMConstr(base_case_trans_cons_mat, gp.hstack((generators, loads)), '>=', -1*thermal_flow_lim, 'Base Case Transimission Min')
base_case_cons_max = lp_model.addMConstr(base_case_trans_cons_mat, gp.hstack((generators, loads)), '<=', thermal_flow_lim, 'Base Case Transimission Max')
contingency_cons_min = lp_model.addMConstr(contingency_trans_cons_mat, gp.hstack((generators, loads)), '>=', -1*thermal_flow_lim_contingencies, 'Contingency Transimission Min')
contingency_cons_max = lp_model.addMConstr(contingency_trans_cons_mat, gp.hstack((generators, loads)), '<=', thermal_flow_lim_contingencies, 'Contingency Transimission Max')

lp_model.update()
lp_model.write('./DA_Clearance.lp')
lp_model.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2927 rows, 27 columns and 69785 nonzeros
Model fingerprint: 0x761ceeaa
Coefficient statistics:
  Matrix range     [1e-04, 1e+00]
  Objective range  [2e+01, 5e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 5e+02]
Presolve removed 2469 rows and 0 columns
Presolve time: 0.02s
Presolved: 458 rows, 255 columns, 12023 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.7262500e+04   1.159054e+02   0.000000e+00      0s
       7    1.7262500e+04   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.04 seconds (0.02 work units)
Optimal objective  1.726250000e+04


## Solutions

In [52]:
power_produced = generators.X
power_served = loads.X

variable_ids = np.hstack([np.char.add('Gen_', generator_node.astype(str)), np.char.add('Load_', load_node.astype(str))])
bus_number = np.hstack([generator_node, load_node])
variable_types = np.hstack([np.full(generator_node.shape[0], 'Generator'), np.full(load_node.shape[0], 'Load')]) 
values = np.hstack([power_produced, power_served])

solutions_df = pd.DataFrame()

solutions_df['Variable ID'] = variable_ids
solutions_df['Bus Number'] = bus_number
solutions_df['Type'] = variable_types
solutions_df['Value'] = values

solutions_df.to_csv('./solutions.csv')